# Notebook 5 — Beyond BM25: Better Query Encoding for Neuroimaging Search

*Hands-on time: ~45 minutes*

BM25 is a bag-of-words model: it matches documents by term frequency/rarity,
with no understanding of *meaning*. A user who types:

> "resting state BOLD on a 3T Siemens"

expects the engine to understand the implied context — that "resting state"
implies TaskName=rest, suffix=bold, that field strength and manufacturer are
structured metadata — but BM25 only does literal token matching.

This notebook explores three concrete upgrade paths, from easiest to most powerful:

| Option | What changes | Effort | Gain |
|--------|-------------|--------|------|
| **A — Better Encoder** | Swap `all-MiniLM-L6-v2` for a larger/domain-aware model | Low | Medium |
| **B — LLM Query Expansion** | Pre-process the query with an LLM before search | Medium | High |
| **C — ELSER** | Replace BM25 `match` with Elasticsearch's learned sparse encoder | Medium | High |
| **D — RRF Fusion** | Combine multiple retrievers with Reciprocal Rank Fusion | Low | High |

**Prerequisites:** Notebooks 1–3 completed.

In [ ]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch, helpers
import warnings
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


ES_HOST = os.environ.get("ES_HOST", "http://localhost:9200")
client = Elasticsearch(ES_HOST, request_timeout=120)
INDEX_NAME = "neuroimaging"

assert client.ping(), f"Cannot reach ES at {ES_HOST}"

# The base index was built with all-mpnet-base-v2 (768d)
BASELINE_MODEL_NAME = "all-mpnet-base-v2"
baseline_model = SentenceTransformer(BASELINE_MODEL_NAME, device='cpu')

print(f"Connected to ES {client.info()['version']['number']}")
print(
    f"Documents in '{INDEX_NAME}': {client.count(index=INDEX_NAME)['count']}")
print(
    f"Baseline model: {BASELINE_MODEL_NAME}  ({baseline_model.get_sentence_embedding_dimension()}d)")
print("Device: CPU-only")


def show_hits(response, fields=None, label="", k=10):
    rows = []
    hits = response["hits"]["hits"] if isinstance(response, dict) else response
    for hit in hits[:k]:
        row = {"_score": round(float(hit.get("_score") or 0), 4)}
        src = hit.get("_source", {})
        if fields:
            for f in fields:
                val = src.get(f, "")
                row[f] = (
                    str(val)[:80] + "…") if f == "description_text" and len(str(val)) > 80 else val
        else:
            row.update({k: v for k, v in src.items() if k not in (
                "metadata_embedding", "specter2_embedding")})
        rows.append(row)
    df = pd.DataFrame(rows)
    if label:
        print(f"\n{'='*60}\n{label}\n{'='*60}")
    return df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents: 4423
Baseline model dims: 384
Device: CPU-only


---
## Option A — Better Sentence Transformer Encoder

The **base index** already uses `all-mpnet-base-v2` (768d), upgraded from the
original `all-MiniLM-L6-v2` (384d).  For neuroimaging/scientific text, a
*domain-aware* model gives meaningfully better embeddings without any changes to
the ES index schema — just re-embed at ingest time.

### Encoder comparison (drop-in, ascending quality)

| Model | Dims | Training domain | When to use |
|-------|------|-----------------|-------------|
| `all-MiniLM-L6-v2` | 384 | General (MS MARCO) | Fast debugging only |
| `all-mpnet-base-v2` ✅ **current** | 768 | General (MS MARCO) | Good baseline |
| `allenai/specter2_base` | 768 | Scientific papers (S2ORC + Semantic Scholar) | **Scientific BIDS text** ← use this |
| `BAAI/bge-large-en-v1.5` | 1024 | General + BEIR fine-tuned | Max retrieval quality |
| `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract` | 768 | PubMed biomedical | Clinical / medical imaging |

> **`specter2_base` vs `specter2`:** `allenai/specter2` is an adapter-augmented
> model that requires the PEFT library and a task-specific LoRA adapter — it cannot
> be loaded with `SentenceTransformer()` directly and will throw a
> `'base_model_name_or_path'` error.  `allenai/specter2_base` is the underlying
> 768-d BERT encoder trained on 8 M scientific paper titles and abstracts; it is
> fully sentence-transformers compatible and is the correct model for producing
> document embeddings.

**Key insight for neuroimaging:** BIDS `description_text` contains terms like
"BOLD", "MP2RAGE", "RepetitionTime", "inversion recovery", "ASL".  A general
encoder trained on news or Wikipedia maps these into generic sub-spaces.
`specter2_base` was trained on scientific literature — it has seen these terms in
context and produces more discriminative embeddings for domain-specific queries.

> **To adopt:** Change `EMBEDDING_DIMS` and the model name in Notebook 1,
> delete and re-create the index, run ingest again.  See **Notebook 08** for a
> full PoC-3 benchmark of specter2_base vs. bge-large.


In [ ]:
# Compare query representations: mpnet (baseline) vs. specter2_base (domain-aware)
#
# allenai/specter2_base — the base BERT encoder trained on 8M scientific paper
# titles/abstracts from Semantic Scholar. 768d, sentence-transformers compatible.
# Already cached in this environment.
#
# NOTE: do NOT use 'allenai/specter2' — that variant requires PEFT task adapters
# and will fail with a 'base_model_name_or_path' KeyError. specter2_base is the
# correct model for producing document and query embeddings.

from scipy.spatial.distance import cosine as cos_dist
import numpy as np
from sentence_transformers import SentenceTransformer

BASELINE_MODEL_NAME = "all-mpnet-base-v2"       # current base index encoder
DOMAIN_MODEL_NAME = "allenai/specter2_base"   # domain-aware scientific encoder

print(f"Loading baseline: {BASELINE_MODEL_NAME}")
baseline_model = SentenceTransformer(BASELINE_MODEL_NAME, device='cpu')
print(f"  dims = {baseline_model.get_sentence_embedding_dimension()}")

try:
    print(f"\nLoading domain model: {DOMAIN_MODEL_NAME}")
    domain_model = SentenceTransformer(DOMAIN_MODEL_NAME, device='cpu')
    print(f"  ✅ dims = {domain_model.get_sentence_embedding_dimension()}")
    DOMAIN_AVAILABLE = True
except Exception as e:
    print(f"  ❌ {DOMAIN_MODEL_NAME} failed: {e}")
    print("  Falling back to baseline (re-run cell to retry download)")
    domain_model = baseline_model
    DOMAIN_MODEL_NAME = BASELINE_MODEL_NAME + " (fallback)"
    DOMAIN_AVAILABLE = False


# ── Cosine similarity between baseline vs. domain model on key queries ────────

test_queries = [
    "resting state fMRI spontaneous brain connectivity",
    "T1 mapping inversion recovery MPRAGE MP2RAGE qMRI",
    "diffusion weighted imaging white matter tractography b-value bvec",
    "arterial spin labeling ASL cerebral blood flow perfusion",
    "Siemens 3T BOLD task activation experimental paradigm",
]

print("\nCosine similarity between baseline and domain-model query vectors:")
print("(lower = models diverge more — domain model encodes differently)")
print()
for q in test_queries:
    v1 = baseline_model.encode(q)
    v2 = domain_model.encode(q)
    sim = 1 - cos_dist(v1, v2)
    print(f"  {q!r:60s}  cosine={sim:.3f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 1.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Loading baseline: all-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
No sentence-transformers model found with name allenai/specter2. Creating a new one with mean pooling.


  dims = 768

Loading domain model: allenai/specter2
  allenai/specter2 not available — Loading a PEFT model requires installing the `peft` package. You can install it via `pip install peft`.
  Falling back to baseline for comparison cells (set local_files_only=False to download)

Cosine similarity between baseline and domain-model query vectors:
(lower = models diverge more — domain model encodes differently)

  'resting state fMRI spontaneous brain connectivity'           cosine=1.000
  'T1 mapping inversion recovery MPRAGE MP2RAGE qMRI'           cosine=1.000
  'diffusion weighted imaging white matter tractography b-value bvec'  cosine=1.000
  'arterial spin labeling ASL cerebral blood flow perfusion'    cosine=1.000
  'Siemens 3T BOLD task activation experimental paradigm'       cosine=1.000


### 1a. Which model better separates functional from structural?

We check whether `specter2` produces a *larger discrimination gap* between
a functional query and irrelevant structural candidates.  Larger gap →
fewer borderline false-positives in the kNN index.


In [ ]:
from itertools import combinations

if not DOMAIN_AVAILABLE:
    print("specter2 not loaded — cannot run comparison (using baseline only).")
else:
    query = "resting state fMRI blood oxygen level dependent BOLD"

    candidates = [
        ("BOLD functional MRI resting state spontaneous brain activity",
         "✅ functional rest"),
        ("fMRI task activation cognitive experiment paradigm",
         "✅ functional task"),
        ("T1-weighted structural anatomical MPRAGE 1 mm isotropic",
         "❌ structural T1w"),
        ("diffusion tensor imaging white matter tractography streamlines",
         "❌ diffusion DWI"),
        ("quantitative T1 map inversion recovery relaxometry",              "❌ qMRI T1map"),
        ("MRI hardware Siemens Prisma 3T bore shim coil",
         "❌ scanner meta"),
    ]

    rows = []
    for text, label in candidates:
        row = {"label": label, "candidate": text[:55]}
        for name, model in [("mpnet", baseline_model), ("specter2", domain_model)]:
            q_vec = model.encode(query, show_progress_bar=False)
            c_vec = model.encode(text,  show_progress_bar=False)
            sim = 1 - cos_dist(q_vec, c_vec)
            row[f"sim_{name}"] = round(sim, 4)
        rows.append(row)

    import pandas as pd
    df = pd.DataFrame(rows).sort_values("sim_specter2", ascending=False)
    display(df)

    # Gap between top-2 relevant and top-1 irrelevant
    rel_scores = {m: [] for m in ["mpnet", "specter2"]}
    irrel_scores = {m: [] for m in ["mpnet", "specter2"]}
    for r in rows:
        for m in ["mpnet", "specter2"]:
            (rel_scores[m] if r["label"].startswith("✅")
             else irrel_scores[m]).append(r[f"sim_{m}"])

    print()
    for m in ["mpnet", "specter2"]:
        gap = min(rel_scores[m]) - max(irrel_scores[m])
        print(
            f"  {m}: min(✅)={min(rel_scores[m]):.4f}  max(❌)={max(irrel_scores[m]):.4f}  gap={gap:+.4f}")

    print()
    print("Positive gap = domain model pushes relevant items above irrelevant ones.")

---
## 1b. Upgrade the Base Index — Add `specter2_embedding`

ES supports adding **new fields** to an existing index via `PUT _mapping` — no
reindex needed.  We add a second `dense_vector` field `specter2_embedding` (768d,
same dimensions as `metadata_embedding`) and then bulk-update every document with
its specter2_base vector.

This approach lets us:
- Keep `metadata_embedding` (mpnet) untouched — PoC-1 and PoC-2 continue to work
- Query *either* embedding field with kNN in the same index
- Compare retrieval quality between the two encoders on identical documents

> The update takes ~2–5 minutes for ~3 000 documents on CPU.


In [ ]:
from tqdm.auto import tqdm

# ── Step 1: add the new field to the mapping (safe to run multiple times) ─────
new_field = {
    "properties": {
        "specter2_embedding": {
            "type": "dense_vector",
            "dims": 768,
            "similarity": "cosine",
            "index_options": {"type": "int8_hnsw"},
        }
    }
}

mapping_resp = client.indices.put_mapping(index=INDEX_NAME, body=new_field)
print(f"Mapping update: {mapping_resp}")

# ── Step 2: check how many docs already have specter2_embedding ───────────────
already = client.count(
    index=INDEX_NAME,
    query={"exists": {"field": "specter2_embedding"}}
)["count"]
total = client.count(index=INDEX_NAME)["count"]
print(f"Docs with specter2_embedding: {already}/{total}")

if already == total:
    print("✅ All documents already have specter2_embedding — skipping bulk update.")
else:
    if not DOMAIN_AVAILABLE:
        print("❌ specter2_base not loaded — cannot populate specter2_embedding.")
        print("   Re-run cell 4 to load the model, then re-run this cell.")
    else:
        print(
            f"Populating specter2_embedding for {total - already} documents ...")

        def fetch_all_docs(index, batch=500):
            docs, from_ = [], 0
            while True:
                resp = client.search(
                    index=index, query={"match_all": {}},
                    size=batch, from_=from_,
                    source_excludes=[
                        "metadata_embedding", "specter2_embedding"]
                )
                hits = resp["hits"]["hits"]
                if not hits:
                    break
                docs.extend(hits)
                from_ += len(hits)
                if from_ >= resp["hits"]["total"]["value"]:
                    break
            return docs

        all_docs = fetch_all_docs(INDEX_NAME)
        print(
            f"Fetched {len(all_docs)} docs — encoding with specter2_base ...")

        ENCODE_BATCH = 32
        actions = []
        for i in tqdm(range(0, len(all_docs), ENCODE_BATCH), desc="Encoding"):
            batch = all_docs[i:i + ENCODE_BATCH]
            texts = [d["_source"].get("description_text", "") for d in batch]
            vecs = domain_model.encode(texts, show_progress_bar=False).tolist()
            for doc, vec in zip(batch, vecs):
                actions.append({
                    "_op_type": "update",
                    "_index":   INDEX_NAME,
                    "_id":      doc["_id"],
                    "doc":      {"specter2_embedding": vec},
                })

        helpers.bulk(client, actions, chunk_size=200)
        client.indices.refresh(index=INDEX_NAME)

        done = client.count(
            index=INDEX_NAME,
            query={"exists": {"field": "specter2_embedding"}}
        )["count"]
        print(f"✅ Done — {done}/{total} documents now have specter2_embedding")

---
## Option B — LLM Query Expansion (Pre-processing)

This is the highest-leverage, zero-re-index change: **before** sending any query to
Elasticsearch, route it through an LLM that understands neuroimaging.

The LLM's job:
1. Expand implicit domain knowledge: *"resting state scan"* → adds *"BOLD, TaskName=rest, rsfMRI"*
2. Normalise synonyms: *"functional MRI"* → *"fMRI, BOLD, bold, suffix=bold"*
3. Surface implied filters: *"Siemens 3T"* → *"Manufacturer=Siemens MagneticFieldStrength=3"*

The expanded text feeds **both** BM25 and the kNN encoder simultaneously.

### Integration paths

| Backend | How | Local? |
|---------|-----|--------|
| **Ollama** (recommended) | `ollama pull llama3` + REST call | ✅ fully local |
| **OpenAI / Groq** | one API call via `openai` library | ❌ sends query to cloud |
| **Rule-based fallback** | synonym dictionary | ✅ no dependencies |

In [25]:
import os
import urllib.request
import json as _json
import re

_OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_URL = _OLLAMA_HOST + "/api/generate"

# ── BIDS corpus context injected into the prompt ──────────────────────────────
_BIDS_SCHEMA_CONTEXT = """
Available BIDS suffixes and their meaning:
  bold   = BOLD fMRI functional scan (Blood Oxygen Level Dependent)
  T1w    = T1-weighted structural anatomical MRI
  T2w    = T2-weighted structural anatomical MRI
  dwi    = Diffusion Weighted Imaging (DWI/DTI)
  phasediff / magnitude1 / magnitude2 = fieldmap for B0 distortion correction
  epi    = EPI fieldmap (Spin-Echo pair)
  UNIT1  = MP2RAGE uniform T1 image
  T1map  = Quantitative R1/T1 relaxation time map (qMRI)
  T2map  = Quantitative T2 relaxation time map
  asl    = Arterial Spin Labeling (perfusion)
  m0scan = M0 reference for ASL normalisation
  pet    = Positron Emission Tomography

Key Elasticsearch index fields (use field=value format in expansions):
  suffix             = BIDS file suffix (see above)
  task / TaskName    = Name of cognitive / experimental task (e.g. rest, stroop)
  MagneticFieldStrength = Field strength in Tesla (1.5, 3, 7)
  Manufacturer / ManufacturersModelName = Scanner brand (Siemens, Philips, GE)
  RepetitionTime     = TR in seconds (fMRI: 1.0–3.0 s; fast EPI: <1.5 s)
  EchoTime           = TE in milliseconds
  InversionTime      = TI for inversion recovery (qMRI: 100–2000 ms)
  FlipAngle          = degrees (VFA / FLASH: multiple angles)
  PhaseEncodingDirection = AP, PA, RL, LR (for fieldmaps and distortion correction)
  InstitutionName    = Hospital or research centre name

Dataset names in this corpus:
  ds001–ds116  = General neuroimaging tasks (fMRI, structural)
  7t_trt       = 7 Tesla test-retest fMRI reliability
  qmri_mp2rage / qmri_irt1 / qmri_mese / qmri_vfa / qmri_mtsat = Quantitative MRI
  asl001–asl005 = Arterial Spin Labeling perfusion studies
  ds000117      = Multi-subject multi-modal (fMRI + MEG)
  eeg_rest_fmri = Simultaneous EEG-fMRI resting state
  genetics_ukbb = Genetics + UK Biobank structural MRI
"""

SYSTEM_PROMPT = (
    "You are a neuroimaging search assistant. Your task is to expand a short user"
    " query into richer BIDS-aware search terms that will find the most relevant"
    " files in an Elasticsearch index of neuroimaging metadata. Following, before"
    " expansion rules, is the actual schema for the BIDS data ingested.\n"
    + _BIDS_SCHEMA_CONTEXT +
    "\nExpansion rules:\n"
    "- Expand abbreviations: BOLD → Blood Oxygen Level Dependent BOLD fMRI,"
    " DWI → Diffusion Weighted Imaging DTI diffusion tensor.\n"
    "- Add BIDS field=value tokens where applicable: 'resting state' →"
    " 'TaskName=rest task=rest rsfMRI default-mode-network'.\n"
    "- Add synonyms for scanner hardware: 'Siemens 3T' →"
    " 'Manufacturer=Siemens MagneticFieldStrength=3'.\n"
    "- The scanner model should compound into the correct BIDS field for it.\n"
    "- Include adjacent modality terms to improve recall.\n"
    "- Use AND and OR and NOT and () - aka prenthesis - boolean operators to structure compound queries.\n"
    "- Use parenthesis profusely(as much as you can) to separate query statements.\n"
    " Don't make assumptions on how they will be separated by the parser. Validate that the queries as outputed"
    " make a complete and valid high-level boolean compounding statement."
    "- Output ONLY a single line of space-separated tokens — no explanation, no JSON, no markdown."
    " "
)


def llm_expand(query: str, model: str = "llama3") -> str | None:
    """Send query to Ollama; return expanded string or None on failure."""
    payload = {
        "model": model,
        "prompt": f"{SYSTEM_PROMPT}\n\nQuery: {query}\nExpanded:",
        "stream": False
    }
    try:
        req = urllib.request.Request(
            OLLAMA_URL, data=_json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=120) as resp:
            return _json.loads(resp.read()).get("response", query).strip()
    except Exception as e:
        print(f"llm encoding failed {e}")
        return None


# ── Deterministic rule-based fallback ─────────────────────────────────────────
SYNONYM_MAP = {
    "resting state":  "resting state rest TaskName=rest task=rest rsfMRI default-mode-network spontaneous",
    "resting-state":  "resting state rest TaskName=rest task=rest rsfMRI",
    "fmri":           "fMRI BOLD bold functional Blood-Oxygen-Level-Dependent suffix=bold",
    "functional":     "functional fMRI BOLD bold suffix=bold activation",
    "diffusion":      "diffusion DWI dwi DTI dti white-matter tractography bvec bval b-value",
    "dti":            "DTI DWI diffusion tensor imaging white matter tractography",
    "anatomical":     "anatomical structural T1w T2w MPRAGE brain morphology",
    "structural":     "structural anatomical T1w MPRAGE MP2RAGE suffix=T1w",
    "t1":             "T1w T1 structural anatomical MPRAGE inversion-recovery",
    "t2":             "T2w T2 structural spin-echo suffix=T2w",
    "mapping":        "quantitative qMRI relaxometry T1map T2map suffix=T1map",
    "quantitative":   "quantitative qMRI relaxometry suffix=T1map InversionTime",
    "mp2rage":        "MP2RAGE UNIT1 suffix=UNIT1 inversion-recovery 3T 7T",
    "asl":            "ASL arterial-spin-labeling suffix=asl m0scan perfusion cerebral-blood-flow",
    "perfusion":      "perfusion ASL arterial-spin-labeling suffix=asl m0scan CBF",
    "fieldmap":       "fieldmap B0 phasediff magnitude1 magnitude2 epi suffix=phasediff distortion-correction",
    "bold":           "BOLD fMRI bold suffix=bold functional Blood-Oxygen-Level-Dependent",
    "3t":             "3T 3Tesla MagneticFieldStrength=3 three-Tesla",
    "1.5t":           "1.5T MagneticFieldStrength=1.5",
    "7t":             "7T 7Tesla MagneticFieldStrength=7 ultra-high-field",
    "siemens":        "Siemens TrioTim Prisma Skyra Verio Manufacturer=Siemens",
    "philips":        "Philips Achieva Intera Manufacturer=Philips",
    "ge":             "GE Discovery Signa Manufacturer=GE",
}


def rule_based_expand(query: str) -> str:
    lower = query.lower()
    extra = [exp for kw, exp in SYNONYM_MAP.items()
             if re.search(r'\b' + re.escape(kw) + r'\b', lower)]
    return (query + " " + " ".join(extra)).strip() if extra else query


def expand_query(query: str) -> tuple[str, str]:
    """Try LLM, fall back to rule-based. Returns (expanded_text, method_name)."""
    result = llm_expand(query)
    if result:
        return result, "llm"
    return rule_based_expand(query), "rule-based"


# Smoke test
for q in [
    "resting state fMRI on a Siemens 3T scanner",
    "quantitative T1 mapping MP2RAGE",
    "ASL perfusion 7T",
]:
    expanded, method = expand_query(q)
    print(f"Original  : {q}")
    print(f"Expanded ({method}):")
    print(f"  {expanded}")
    print()

Original  : resting state fMRI on a Siemens 3T scanner
Expanded (llm):
  TaskName=rest task=rest rsfMRI default-mode-network AND suffix=BOLD AND MagneticFieldStrength=3 AND Manufacturer=Siemens AND (InstitutionName=("general neuroimaging tasks" OR "7t_trt" OR "qmri_irt1") OR InstitutionName="genetics_ukbb")

Original  : quantitative T1 mapping MP2RAGE
Expanded (llm):
  T1w UNIT1 suffix=T1map TaskName=quantitative Mapping=MP2RAGE AND MagneticFieldStrength=7 AND Manufacturer=(Siemens|Philips|GE) AND InstitutionName=(qmri_mp2rage|qmri_irt1|qmri_mese|qmri_vfa|qmri_mtsat)

Original  : ASL perfusion 7T
Expanded (llm):
  asl=asl001-005 AND MagneticFieldStrength=7 AND Manufacturer=(Siemens OR Philips OR GE)



In [26]:
# Compare BM25 match: original query vs. expanded query

def bm25_search(query_text, size=5):
    return client.search(
        index=INDEX_NAME,
        query={"match": {"description_text": query_text}},
        size=size
    )


test_queries_b = [
    "resting state fMRI",
    "diffusion tensor imaging white matter",
    "anatomical Siemens 3T",
]

for q in test_queries_b:
    expanded, method = expand_query(q)
    r_orig = bm25_search(q)
    r_exp = bm25_search(expanded)

    h_orig = r_orig["hits"]["total"]["value"]
    h_exp = r_exp["hits"]["total"]["value"]
    s_orig = r_orig["hits"]["hits"][0]["_score"] if r_orig["hits"]["hits"] else 0
    s_exp = r_exp["hits"]["hits"][0]["_score"] if r_exp["hits"]["hits"] else 0

    print(f"Query: {q!r}")
    print(f"  Original  → hits={h_orig:4d}, top_score={s_orig:.3f}")
    print(f"  Expanded  → hits={h_exp:4d},  top_score={s_exp:.3f}  [{method}]")
    print()

Query: 'resting state fMRI'
  Original  → hits= 412, top_score=2.304
  Expanded  → hits=3691,  top_score=24.132  [llm]

Query: 'diffusion tensor imaging white matter'
  Original  → hits=  83, top_score=24.510
  Expanded  → hits=1807,  top_score=26.323  [llm]

Query: 'anatomical Siemens 3T'
  Original  → hits=1272, top_score=6.242
  Expanded  → hits=1770,  top_score=9.871  [llm]



In [27]:
# Expanded hybrid search — expansion benefits BOTH BM25 and kNN simultaneously
#
# When specter2_base is loaded (DOMAIN_AVAILABLE=True), kNN queries the
# specter2_embedding field (domain-aware 768d vectors).
# When not loaded, falls back to metadata_embedding (mpnet 768d).

FIELDS = [
    "dataset",
    "datatype",
    "suffix",
    "TaskName",
    "MagneticFieldStrength",
    "Manufacturer",
    "ManufacturersModelName",
    "description_text",
]

# Choose encoder and field based on what's loaded
if DOMAIN_AVAILABLE:
    _knn_encoder = domain_model
    _knn_field = "specter2_embedding"
    _knn_label = "specter2_base"
else:
    _knn_encoder = baseline_model
    _knn_field = "metadata_embedding"
    _knn_label = "mpnet (fallback)"

print(f"kNN encoder: {_knn_label}  →  field: {_knn_field}")


def hybrid_expanded(query: str, k: int = 10):
    expanded, method = expand_query(query)
    response = client.search(
        index=INDEX_NAME,
        query={"match": {"description_text": {"query": expanded, "boost": 0.3}}},
        knn={
            "field":          _knn_field,
            "query_vector":   _knn_encoder.encode(expanded).tolist(),
            "k":              k,
            "num_candidates": k * 20,
            "boost":          0.7,
        },
        size=k,
        source_excludes=["metadata_embedding", "specter2_embedding"],
    )
    print(f"Query    : {query!r}")
    print(f"Expanded : {expanded!r}  [{method}]")
    print(f"kNN      : {_knn_field} ({_knn_label})")
    return response


resp = hybrid_expanded("resting state functional MRI on Siemens 3T")
display(show_hits(resp, FIELDS))

kNN encoder: specter2_base  →  field: specter2_embedding
Query    : 'resting state functional MRI on Siemens 3T'
Expanded : 'TaskName=rest task=rest rsfMRI default-mode-network AND suffix=bold AND MagneticFieldStrength=3 AND Manufacturer=Siemens AND InstitutionName=(any) AND RepetitionTime:(1-3) AND NOT pet'  [llm]
kNN      : specter2_embedding (specter2_base)


,_score,dataset,datatype,suffix,TaskName,MagneticFieldStrength,Manufacturer,ManufacturersModelName,description_text
0,5.2594,eeg_rest_fmri,func,bold,rest,1.5,Siemens,Avanto,BOLD functional MRI Blood Oxygen Level Depende...
1,5.2594,eeg_rest_fmri,func,bold,rest,1.5,Siemens,Avanto,BOLD functional MRI Blood Oxygen Level Depende...
2,5.2594,eeg_rest_fmri,func,bold,rest,1.5,Siemens,Avanto,BOLD functional MRI Blood Oxygen Level Depende...
3,4.5782,eeg_rest_fmri,anat,T1w,NaN,1.5,Siemens,Avanto,T1-weighted anatomical structural MRI (structu...
4,4.5782,eeg_rest_fmri,anat,T1w,NaN,1.5,Siemens,Avanto,T1-weighted anatomical structural MRI (structu...
5,4.5782,eeg_rest_fmri,dwi,dwi,NaN,1.5,Siemens,Avanto,diffusion-weighted imaging DWI DTI tractograph...
6,4.5782,eeg_rest_fmri,dwi,dwi,NaN,1.5,Siemens,Avanto,diffusion-weighted imaging DWI DTI tractograph...
7,4.5782,eeg_rest_fmri,anat,T1w,NaN,1.5,Siemens,Avanto,T1-weighted anatomical structural MRI (structu...
8,4.5782,eeg_rest_fmri,anat,T1w,NaN,1.5,Siemens,Avanto,T1-weighted anatomical structural MRI (structu...
9,4.5782,eeg_rest_fmri,dwi,dwi,NaN,1.5,Siemens,Avanto,diffusion-weighted imaging DWI DTI tractograph...


---
## Option C — ELSER: Elastic Learned Sparse Encoder

ELSER is Elasticsearch's own **learned sparse retrieval** model.
Instead of BM25's hand-crafted TF-IDF statistics, ELSER uses a
transformer to assign vocabulary token weights that encode semantic meaning.

**Why it beats BM25:**
- Handles synonyms (`resting state` ↔ `rest`, `fMRI` ↔ `BOLD`) automatically
- Understands token importance in context, not just raw frequency
- No query expansion needed — semantic understanding is baked in

**vs. dense kNN:** sparse → fast, explainable, works well with BM25-style indexing

**Requirements:** ES 8.6+ default distribution (included, needs ML node allocation)

In [28]:
# Check whether the ELSER inference endpoint is deployed (ES 9.x Inference API)
try:
    resp = client.inference.get(inference_id="elser-model-2")
    endpoints = resp.get("endpoints", [])
    if endpoints:
        for ep in endpoints:
            print(
                f"ELSER ready: inference_id={ep['inference_id']}  service={ep.get('service')}")
            ss = ep.get("service_settings", {})
            print(
                f"  allocations={ss.get('num_allocations')}  threads={ss.get('num_threads')}")
    else:
        print("Endpoint registered but no active allocations — model may still be loading.")
except Exception as e:
    print(f"ELSER not deployed: {e}")
    print()
    print("To deploy, run the next cell.")
    print("Note: ELSER requires a Trial or Enterprise license.")
    print("  Start 30-day trial: POST /_license/start_trial?acknowledge=true")
    print("  or: Kibana → Stack Management → License Management → Start Trial")

ELSER ready: inference_id=elser-model-2  service=elasticsearch
  allocations=1  threads=1


In [29]:
# Deploy ELSER via the Inference API (ES 9.3 syntax).
#
# REQUIRES a Trial or Enterprise license — activate a 30-day trial first:
#   curl -X POST "http://localhost:9200/_license/start_trial?acknowledge=true"
#   or: Kibana → Stack Management → License Management → Start Trial
#
# Also confirm xpack.ml.enabled=true in docker-compose.yml (already set there).

try:
    client.inference.put(
        task_type="sparse_embedding",
        inference_id="elser-model-2",
        inference_config={
            "service": "elser",
            "service_settings": {"num_allocations": 1, "num_threads": 1}
        }
    )
    print("ELSER endpoint registered (~250 MB model download on first use).")
    print("Wait 1–2 min for the model to fully load, then re-run the check cell above.")
except Exception as e:
    print(f"Error: {e}")
    if "license" in str(e).lower():
        print()
        print("License error — activate a trial first:")
        print(
            "  curl -X POST 'http://localhost:9200/_license/start_trial?acknowledge=true'")
        print("Then re-run this cell.")

Error: BadRequestError(400, 'status_exception', 'Inference endpoint IDs must be unique. Requested inference endpoint ID [elser-model-2] matches existing trained model ID(s) but must not.')


In [30]:
# Once ELSER is deployed:
# 1. Add a sparse_vector field to the index mapping
# 2. Create an ingest pipeline that calls ELSER inference
# 3. Re-index documents through the pipeline

ELSER_PIPELINE = {
    "description": "ELSER sparse embedding pipeline",
    "processors": [{
        "inference": {
            "model_id": "elser-model-2",
            "input_output": [{
                "input_field": "description_text",
                "output_field": "elser_embedding"
            }]
        }
    }]
}

print("Ingest pipeline to create:")
print(json.dumps(ELSER_PIPELINE, indent=2))

# Replace BM25 match queries with text_expansion:
ELSER_QUERY = {
    "text_expansion": {
        "elser_embedding": {
            "model_id": "elser-model-2",
            "model_text": "resting state fMRI on a Siemens 3T scanner"
        }
    }
}
print("\nReplacement query DSL (swap this for the BM25 match clause):")
print(json.dumps(ELSER_QUERY, indent=2))

Ingest pipeline to create:
{
  "description": "ELSER sparse embedding pipeline",
  "processors": [
    {
      "inference": {
        "model_id": "elser-model-2",
        "input_output": [
          {
            "input_field": "description_text",
            "output_field": "elser_embedding"
          }
        ]
      }
    }
  ]
}

Replacement query DSL (swap this for the BM25 match clause):
{
  "text_expansion": {
    "elser_embedding": {
      "model_id": "elser-model-2",
      "model_text": "resting state fMRI on a Siemens 3T scanner"
    }
  }
}


---
## Option D — Reciprocal Rank Fusion (RRF)

RRF is a score-free fusion strategy: instead of manually tuning boost weights,
it combines the *ranks* from each retriever:

```
rrf_score(d) = sum( 1 / (k + rank_i(d)) )    k=60 default
```

Benefits over weighted hybrid:
- Zero calibration required
- Robust when one retriever returns no results
- Consistently outperforms manual boosting in benchmarks

Available natively in ES 8.8+.

In [31]:
# Check if RRF is available
version_str = client.info()["version"]["number"]
major, minor = map(int, version_str.split(".")[:2])
rrf_available = (major > 8) or (major == 8 and minor >= 8)
print(f"ES version: {version_str}")
print(f"RRF available: {rrf_available}")
if not rrf_available:
    print("→ Upgrade: docker compose down -v && docker compose up -d")
    print("  docker-compose.yml already targets ES 9.3.0 which supports RRF.")

ES version: 9.3.0
RRF available: True


In [37]:
# RRF: BM25 + kNN with no boost tuning.
# Available on all license tiers in ES 9.3 — no special config required.
# Using specter2_embedding (domain-aware) when available.

query_text = "resting state fMRI Siemens 3T"

knn_field = "specter2_embedding" if DOMAIN_AVAILABLE else "metadata_embedding"
knn_encoder = domain_model if DOMAIN_AVAILABLE else baseline_model
knn_label = "specter2_base" if DOMAIN_AVAILABLE else "mpnet (fallback)"

response = client.search(
    index=INDEX_NAME,
    query={"match": {"description_text": query_text}},
    knn={
        "field":          knn_field,
        "query_vector":   knn_encoder.encode(query_text).tolist(),
        "k":              10,
        "num_candidates": 200,
    },
    rank={"rrf": {"rank_window_size": 50, "rank_constant": 60}},
    size=10,
    source_excludes=["metadata_embedding", "specter2_embedding"],
)
print(f'RRF results for: "{query_text}"  (kNN field: {knn_field})')
display(show_hits(response, FIELDS))

RRF results for: "resting state fMRI Siemens 3T"  (kNN field: specter2_embedding)


/tmp/ipykernel_198912/3631115163.py:11: ElasticsearchWarning: Deprecated field [rank] used, replaced by [retriever]
  response = client.search(


,_score,dataset,datatype,suffix,TaskName,MagneticFieldStrength,Manufacturer,ManufacturersModelName,description_text
0,0.0164,7t_trt,func,bold,Rest,NaN,NaN,None,BOLD functional MRI Blood Oxygen Level Depende...
1,0.0164,qmri_tb1tfl,fmap,TB1TFL,NaN,3.0,Siemens,None,turbo-flash B1 transmit field mapping | 3 Tesl...
2,0.0161,7t_trt,func,bold,Rest,NaN,NaN,None,BOLD functional MRI Blood Oxygen Level Depende...
3,0.0161,qmri_tb1tfl,fmap,TB1TFL,NaN,3.0,Siemens,None,turbo-flash B1 transmit field mapping | 3 Tesl...
4,0.0159,7t_trt,func,bold,Rest,NaN,NaN,None,BOLD functional MRI Blood Oxygen Level Depende...
5,0.0159,qmri_mtsat,anat,MTS,NaN,3.0,Siemens,None,magnetization transfer saturation weighted ima...
6,0.0156,7t_trt,func,bold,Rest,NaN,NaN,None,BOLD functional MRI Blood Oxygen Level Depende...
7,0.0156,qmri_mtsat,anat,MTS,NaN,3.0,Siemens,None,magnetization transfer saturation weighted ima...
8,0.0154,7t_trt,func,bold,Rest,NaN,NaN,None,BOLD functional MRI Blood Oxygen Level Depende...
9,0.0154,qmri_mtsat,anat,MTS,NaN,3.0,Siemens,None,magnetization transfer saturation weighted ima...


In [38]:
# Full stack: LLM expansion + specter2 kNN + RRF
#
# BM25 (match) is used as the sparse leg.
# kNN uses specter2_embedding (domain-aware 768d) when specter2_base is loaded,
# or metadata_embedding (mpnet 768d) as fallback.
#
# To substitute ELSER as the sparse retriever, deploy the inference endpoint
# (cell 13 above — requires Trial/Enterprise license), then replace the "query"
# clause with a "text_expansion" against the elser_embedding field.

knn_field = "specter2_embedding" if DOMAIN_AVAILABLE else "metadata_embedding"
knn_encoder = domain_model if DOMAIN_AVAILABLE else baseline_model
knn_label = "specter2_base" if DOMAIN_AVAILABLE else "mpnet (fallback)"

expanded, method = expand_query(query_text)
response = client.search(
    index=INDEX_NAME,
    query={"match": {"description_text": expanded}},
    knn={
        "field":          knn_field,
        "query_vector":   knn_encoder.encode(expanded).tolist(),
        "k":              10,
        "num_candidates": 200,
    },
    rank={"rrf": {"rank_window_size": 50, "rank_constant": 60}},
    size=10,
    source_excludes=["metadata_embedding", "specter2_embedding"],
)
print(f'Full stack (expansion={method}, kNN={knn_label}):')
print(f'  "{expanded[:90]}…"')
display(show_hits(response, FIELDS))

Full stack (expansion=llm, kNN=specter2_base):
  "TaskName=rest task=rest rsfMRI default-mode-network AND suffix=bold AND MagneticFieldStren…"


/tmp/ipykernel_198912/280242699.py:16: ElasticsearchWarning: Deprecated field [rank] used, replaced by [retriever]
  response = client.search(


,_score,dataset,datatype,suffix,TaskName,MagneticFieldStrength,Manufacturer,ManufacturersModelName,description_text
0,0.0164,2d_mb_pcasl,fmap,epi,NaN,3.0,Siemens,Prisma_fit,EPI fieldmap blip-up blip-down distortion corr...
1,0.0164,eeg_rest_fmri,func,bold,rest,1.5,Siemens,Avanto,BOLD functional MRI Blood Oxygen Level Depende...
2,0.0161,7t_trt,fmap,magnitude1,NaN,NaN,NaN,NaN,magnitude image 1 for fieldmap estimation (fie...
3,0.0161,eeg_rest_fmri,func,bold,rest,1.5,Siemens,Avanto,BOLD functional MRI Blood Oxygen Level Depende...
4,0.0159,7t_trt,fmap,magnitude1,NaN,NaN,NaN,NaN,magnitude image 1 for fieldmap estimation (fie...
5,0.0159,eeg_rest_fmri,func,bold,rest,1.5,Siemens,Avanto,BOLD functional MRI Blood Oxygen Level Depende...
6,0.0156,7t_trt,fmap,magnitude1,NaN,NaN,NaN,NaN,magnitude image 1 for fieldmap estimation (fie...
7,0.0156,eeg_rest_fmri,anat,T1w,NaN,1.5,Siemens,Avanto,T1-weighted anatomical structural MRI (structu...
8,0.0154,7t_trt,fmap,magnitude1,NaN,NaN,NaN,NaN,magnitude image 1 for fieldmap estimation (fie...
9,0.0154,eeg_rest_fmri,anat,T1w,NaN,1.5,Siemens,Avanto,T1-weighted anatomical structural MRI (structu...


---
## Summary — Recommended Migration Path

```
Current stack
  query ──► BM25 match ──────────────────► score  }
         └► all-MiniLM-L6-v2 ► kNN ───────► score  } weighted sum → results

Recommended stack
  query ──► LLM expansion ──► BM25 / ELSER ──► rank  }
                           └► domain encoder ► kNN  ─► RRF → results
```

### Step-by-step upgrade (least to most effort)

1. **Immediate (zero re-index):** wrap every search call with `expand_query()`.
   Both BM25 and kNN benefit immediately. Use Ollama + llama3 locally for best results,
   or the built-in synonym fallback for zero dependencies.

2. **Low-effort re-index:** swap `all-MiniLM-L6-v2` for `BAAI/bge-large-en-v1.5`
   or `allenai/specter2` in Notebook 1. Change `EMBEDDING_DIMS` to 1024 or 768,
   delete the index, re-run ingest.

3. **Upgrade ES to 9.3** (docker-compose.yml already targets this): enables native RRF,
   improved ELSER, and `semantic_text` which auto-chunks + auto-embeds on ingest.

4. **Deploy ELSER** via Kibana ML Management. Add `sparse_vector` field + ingest pipeline.
   Use `text_expansion` query instead of `match` — this is the true BM25 replacement.

5. **Full stack:** LLM expansion → ELSER sparse + domain-encoder dense → RRF.
   State-of-the-art neural retrieval, no fine-tuning required.